# Layout-Aware Form Understanding with LayoutLMv3Fine-tuning **Microsoft LayoutLMv3** on the **FUNSD** dataset for named entity recognition across scanned form documents — classifying each word token as a **Header**, **Question**, or **Answer** using joint text, layout, and image encoding.This notebook covers the full pipeline: data inspection, class-imbalance-aware training, evaluation, correctly word-aligned prediction visualization, OCR-based inference on arbitrary real-world documents (not just pre-annotated FUNSD samples), and exporting the model for serving via the FastAPI app in `app/`.

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q transformers datasets seqeval pillow torch torchvision pytesseract!apt-get -qq install -y tesseract-ocr > /dev/null  # needed for OCR-based inference on raw images (Section 11)

## 2. Load the FUNSD DatasetFUNSD contains **199 real scanned forms** (149 train / 50 test) annotated with bounding boxes and NER labels.Each sample includes: `tokens`, `bboxes` (normalized 0-1000), `ner_tags`, and the document `image`.

In [ ]:
from datasets import load_datasetdataset = load_dataset("nielsr/funsd-layoutlmv3")print(dataset)print("\nSample keys:", dataset["train"][0].keys())

## 3. Check Label DistributionBefore training, look at how balanced the classes actually are — this determines whether we need tocorrect for class imbalance later. (Spoiler: HEADER is heavily underrepresented, which is why theoriginal baseline scored 0.59 F1 on it versus ~0.90 for the other two classes.)

In [ ]:
from collections import Counterlabel_list = [    "O",    "B-HEADER", "I-HEADER",    "B-QUESTION", "I-QUESTION",    "B-ANSWER", "I-ANSWER",]id2label = {i: l for i, l in enumerate(label_list)}label2id = {l: i for i, l in enumerate(label_list)}counts = Counter()for ex in dataset["train"]:    counts.update(ex["ner_tags"])print("Token counts per class (train split):")for i, name in id2label.items():    print(f"  {name:12s} {counts[i]:5d}")header_tokens = counts[label2id["B-HEADER"]] + counts[label2id["I-HEADER"]]question_tokens = counts[label2id["B-QUESTION"]] + counts[label2id["I-QUESTION"]]print(f"\nHEADER tokens: {header_tokens} vs QUESTION tokens: {question_tokens} "      f"({question_tokens / max(header_tokens,1):.1f}x more QUESTION tokens)")

## 4. Visualize Raw DataBefore training, it helps to visualize what the dataset actually looks like — each token mapped to its bounding box on the document image.

In [ ]:
from PIL import Image, ImageDrawimport matplotlib.pyplot as pltsample = dataset["train"][0]image = sample["image"].convert("RGB")draw = ImageDraw.Draw(image)for box in sample["bboxes"]:    draw.rectangle(box, outline="red", width=2)plt.figure(figsize=(10, 12))plt.imshow(image)plt.axis("off")plt.title("FUNSD Sample — Bounding Boxes")plt.savefig("sample_visualized.png", bbox_inches="tight")plt.show()print("Saved!")

## 5. Define Label SchemaWe use BIO (Beginning-Inside-Outside) tagging:- `B-` prefix = beginning of an entity span- `I-` prefix = continuation of an entity span- `O` = not an entity(Schema already loaded in Section 3 above so the imbalance check could use it.)

In [ ]:
print(id2label)

## 6. Preprocess with LayoutLMv3ProcessorThe processor handles:- **Tokenization** of words into subword tokens- **Bounding box alignment** — each subword token inherits its word's bounding box- **Image patching** — the document image is divided into patches for visual encoding- **Label alignment** — only the first subword of each word gets a real label; continuations get `-100` (ignored in loss)We set `apply_ocr=False` since FUNSD already provides tokenized text.

In [ ]:
from transformers import LayoutLMv3Processorprocessor = LayoutLMv3Processor.from_pretrained(    "microsoft/layoutlmv3-base", apply_ocr=False)def preprocess(batch):    images = [img.convert("RGB") for img in batch["image"]]    encoding = processor(        images,        batch["tokens"],        boxes=batch["bboxes"],        word_labels=batch["ner_tags"],        truncation=True,        padding="max_length",        max_length=512,    )    return encodingtrain_dataset = dataset["train"].map(    preprocess, batched=True,    remove_columns=dataset["train"].column_names)test_dataset = dataset["test"].map(    preprocess, batched=True,    remove_columns=dataset["test"].column_names)train_dataset.set_format("torch")test_dataset.set_format("torch")print("Preprocessing done!")print("Train size:", len(train_dataset))print("Test size:", len(test_dataset))

## 7. Load LayoutLMv3 ModelWe load `microsoft/layoutlmv3-base` (125M parameters) and add a token classification head on top with 7 output classes (our BIO label set).

In [ ]:
from transformers import LayoutLMv3ForTokenClassificationmodel = LayoutLMv3ForTokenClassification.from_pretrained(    "microsoft/layoutlmv3-base",    num_labels=len(label_list),    id2label=id2label,    label2id=label2id)print("Model loaded!")

## 8. Handle Class Imbalance (fix for weak HEADER performance)The original baseline trained with plain unweighted cross-entropy, which is exactly why HEADER(the rarest class) lagged 30 F1 points behind ANSWER/QUESTION — the loss barely penalizedmisclassifying it. We compute inverse-frequency class weights from the training set and use themin a weighted cross-entropy loss via a custom `Trainer` subclass, so rare classes contribute moreto the gradient.

In [ ]:
import torchimport torch.nn as nnfrom transformers import Trainerclass_weights = torch.ones(len(label_list))total = sum(counts.values())for i in range(len(label_list)):    freq = counts[i] / total if counts[i] > 0 else 1e-6    class_weights[i] = 1.0 / freqclass_weights = class_weights / class_weights.sum() * len(label_list)  # normalizeprint("Class weights:", {id2label[i]: round(w.item(), 3) for i, w in enumerate(class_weights)})class WeightedTrainer(Trainer):    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):        labels = inputs.pop("labels")        outputs = model(**inputs)        logits = outputs.logits        loss_fct = nn.CrossEntropyLoss(            weight=class_weights.to(logits.device), ignore_index=-100        )        loss = loss_fct(logits.view(-1, len(label_list)), labels.view(-1))        return (loss, outputs) if return_outputs else loss

## 9. Define Evaluation MetricWe use **seqeval** for span-level F1 evaluation — it correctly handles BIO tags by evaluating complete entity spans rather than individual tokens.  This is the standard metric for NER tasks.

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_scoreimport numpy as npdef compute_metrics(p):    predictions, labels = p    predictions = np.argmax(predictions, axis=2)    true_labels = [        [id2label[l] for l in label if l != -100]        for label in labels    ]    true_preds = [        [id2label[pred] for pred, l in zip(preds_row, label) if l != -100]        for preds_row, label in zip(predictions, labels)    ]    return {        "f1": f1_score(true_labels, true_preds),        "precision": precision_score(true_labels, true_preds),        "recall": recall_score(true_labels, true_preds),    }

## 10. Fine-tune the ModelTraining configuration:- **10 epochs**, best checkpoint auto-loaded at the end (selected by F1)- **Learning rate**: 1e-5 (standard for fine-tuning large transformers)- **Batch size**: 2 (constrained by GPU memory for a 125M param model)- **Weighted loss** (Section 8) to correct the HEADER class imbalance

In [ ]:
from transformers import TrainingArgumentsargs = TrainingArguments(    output_dir="layoutlmv3-funsd",    num_train_epochs=10,    per_device_train_batch_size=2,    per_device_eval_batch_size=2,    learning_rate=1e-5,    eval_strategy="epoch",    save_strategy="epoch",    load_best_model_at_end=True,    metric_for_best_model="f1",    logging_steps=10,    report_to="none",)trainer = WeightedTrainer(    model=model,    args=args,    train_dataset=train_dataset,    eval_dataset=test_dataset,    compute_metrics=compute_metrics,)trainer.train()

## 11. Evaluate — Final ResultsRunning full evaluation on the 50-document test set with per-class precision, recall, and F1.

In [ ]:
results = trainer.evaluate()print("\n=== FINAL RESULTS ===")print(f"F1 Score: {results['eval_f1']:.4f}")predictions = trainer.predict(test_dataset)preds = np.argmax(predictions.predictions, axis=2)labels = predictions.label_idstrue_labels = [    [id2label[l] for l in label if l != -100]    for label in labels]true_preds = [    [id2label[pred] for pred, l in zip(preds_row, label) if l != -100]    for preds_row, label in zip(preds, labels)]print("\n=== CLASSIFICATION REPORT ===")print(classification_report(true_labels, true_preds))

### Baseline Results (unweighted loss, prior run)| Entity | Precision | Recall | F1 ||--------|-----------|--------|----|| ANSWER | 0.90 | 0.91 | **0.91** || QUESTION | 0.88 | 0.91 | **0.89** || HEADER | 0.57 | 0.61 | **0.59** || **Overall** | **0.87** | **0.89** | **0.8791** |**Observations from the baseline:**- ANSWER and QUESTION entities are learned with high confidence — these are the most structurally consistent across forms.- HEADER F1 (0.59) lagged because it has ~9x fewer training tokens than QUESTION and highly variable visual formatting.- Overall F1 of **87.91%** is competitive with published LayoutLMv3-base baselines on FUNSD.**After adding weighted loss (Section 8):** re-run the classification report above and paste theupdated numbers here — the weighting is expected to close most of the HEADER precision/recall gapat a small cost to overall accuracy (a real, documentable trade-off worth discussing in an interview,not hidden).

## 12. Save Model for ServingExport the fine-tuned model + processor so the FastAPI service in `app/api.py` can load them withoutdepending on this notebook.

In [ ]:
SAVE_DIR = "./model"trainer.save_model(SAVE_DIR)processor.save_pretrained(SAVE_DIR)print(f"Model and processor saved to {SAVE_DIR}/")print("Copy this folder next to app/ and run: uvicorn app.api:app --reload")

## 13. Visualize Predictions on Test Document (word-aligned)**Bug fix from the original notebook:** predictions come out of the model per *subword token*(up to 512 of them), while `tokens`/`bboxes` in the raw sample are per *word*. The original codezipped `tokens`/`boxes` directly against `preds[1:]` positionally, which silently misaligns as soonas any word splits into more than one subword (very common for LayoutLMv3's tokenizer). This versionuses `encoding.word_ids()` to correctly map each *word* to its first subword's prediction.

In [ ]:
sample = dataset["test"][0]image = sample["image"].convert("RGB")draw = ImageDraw.Draw(image)COLORS = {    "B-HEADER": "blue", "I-HEADER": "blue",    "B-QUESTION": "red", "I-QUESTION": "red",    "B-ANSWER": "green", "I-ANSWER": "green",}encoding = processor(    image, sample["tokens"],    boxes=sample["bboxes"],    return_tensors="pt")word_ids = encoding.word_ids(batch_index=0)encoding = {k: v.to(model.device) for k, v in encoding.items() if k != "word_ids"}with torch.no_grad():    outputs = model(**encoding)pred_ids = outputs.logits.argmax(-1).squeeze().tolist()# Map each word to the prediction of its FIRST subword token only (correct alignment)word_to_pred = {}for token_idx, word_idx in enumerate(word_ids):    if word_idx is not None and word_idx not in word_to_pred:        word_to_pred[word_idx] = pred_ids[token_idx]for word_idx, box in enumerate(sample["bboxes"]):    label = id2label[word_to_pred.get(word_idx, 0)]    color = COLORS.get(label, "gray")    draw.rectangle(box, outline=color, width=2)plt.figure(figsize=(10, 12))plt.imshow(image)plt.axis("off")plt.title("Predictions: Blue=Header | Red=Question | Green=Answer")plt.savefig("predictions_visualized.png", bbox_inches="tight")plt.show()

## 14. Real-World Inference (OCR-based, no pre-existing boxes)FUNSD test samples come with words and boxes already extracted. **Real customer documents don't.**This section demonstrates the model on an arbitrary image using Tesseract OCR to get words + boxesfirst — this is the code path that actually matters for a deployed product, and it's what`app/inference.py` wraps into a reusable function for the API.

In [ ]:
import pytesseractdef ocr_words_and_boxes(pil_image):    """Run Tesseract OCR and return (words, normalized_boxes) in LayoutLMv3's 0-1000 format."""    w, h = pil_image.size    data = pytesseract.image_to_data(pil_image, output_type=pytesseract.Output.DICT)    words, boxes = [], []    for i in range(len(data["text"])):        text = data["text"][i].strip()        if not text:            continue        x, y, bw, bh = data["left"][i], data["top"][i], data["width"][i], data["height"][i]        box = [            int(1000 * x / w), int(1000 * y / h),            int(1000 * (x + bw) / w), int(1000 * (y + bh) / h),        ]        words.append(text)        boxes.append(box)    return words, boxesdef predict_document(pil_image, model, processor, id2label):    pil_image = pil_image.convert("RGB")    words, boxes = ocr_words_and_boxes(pil_image)    if not words:        return []    encoding = processor(pil_image, words, boxes=boxes, return_tensors="pt", truncation=True)    word_ids = encoding.word_ids(batch_index=0)    inputs = {k: v.to(model.device) for k, v in encoding.items() if k != "word_ids"}    with torch.no_grad():        logits = model(**inputs).logits    pred_ids = logits.argmax(-1).squeeze().tolist()    word_to_pred = {}    for token_idx, word_idx in enumerate(word_ids):        if word_idx is not None and word_idx not in word_to_pred:            word_to_pred[word_idx] = pred_ids[token_idx]    return [        {"word": w, "box": b, "label": id2label[word_to_pred.get(i, 0)]}        for i, (w, b) in enumerate(zip(words, boxes))    ]# Demo on a fresh FUNSD test image, but through the OCR path instead of ground-truth tokens/boxes,# i.e. exactly what happens with an unseen document that has no pre-computed annotations.demo_image = dataset["test"][1]["image"].convert("RGB")results = predict_document(demo_image, model, processor, id2label)for r in results[:15]:    print(r)

## 15. Next Steps: Serving as an APIThe model saved in Section 12 is loaded by `app/inference.py` (same OCR-based prediction logic asSection 14, refactored into a module) and exposed over HTTP by `app/api.py`:```bashuvicorn app.api:app --reload --port 8000````POST /predict` accepts an image file and returns per-word labels + boxes as JSON, plus a base64rendered visualization — ready to be called from the demo page in `web/index.html` or any frontend.